#Initialization and Loading

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import *
from pyspark import *
import pandas as pd
import numpy as np

import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger("crm_sales_details_pipeline")

logger.info("Starting Silver Layer Transformation")


#Read from Bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_sales_details")
logger.info(f"Number of rows (raw): {df.count()}")
logger.info(f"Columns: {df.columns}")
df.display()

#Data quality

In [0]:
# Null check
null_counts = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

logger.info("Null counts:")
null_counts.show()

# Duplicate check
dup_count = df.count() - df.dropDuplicates().count()
logger.info(f"Duplicate rows: {dup_count}")

#Data Transformations

##Trimming

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, trim(col(field.name)))

logger.info("String columns trimmed")

##Cleaning dates

In [0]:
logger.info("Casting sales date columns ('sls_order_dt', 'sls_ship_dt', 'sls_due_dt') to DateType with validation")


df = (
    df
    .withColumn(
        "sls_order_dt",
        when(
            (col("sls_order_dt") == 0) | (length(col("sls_order_dt")) != 8),
            None
        ).otherwise(to_date(col("sls_order_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_ship_dt",
        when(
            (col("sls_ship_dt") == 0) | (length(col("sls_ship_dt")) != 8),
            None
        ).otherwise(to_date(col("sls_ship_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_due_dt",
        when(
            (col("sls_due_dt") == 0) | (length(col("sls_due_dt")) != 8),
            None
        ).otherwise(to_date(col("sls_due_dt").cast("string"), "yyyyMMdd"))
    )
)

##Sales and Price correction

In [0]:
df = (
    df
    .withColumn(
        "sls_price",
        when(
            (col("sls_price").isNull()) | (col("sls_price") <= 0),
            when(
                col("sls_quantity") != 0,
                col("sls_sales") / col("sls_quantity")
            ).otherwise(None)
        ).otherwise(col("sls_price"))
    )
)

logger.info("Calculating 'sls_price' where null or <= 0 using sls_sales / sls_quantity")

##Renaming columns

In [0]:

RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

logger.info("Columns renamed successfully!")

##Data Quality check

In [0]:

df.limit(10).display()

#Writing into Silver Table

In [0]:
logger.info("Writing data to Silver table")

(
    df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.crm_sales")
)

logger.info("Write completed successfully")